In [3]:
def is_variable(x):
    return isinstance(x, str) and x.startswith("")

def unify(x, y, subs=None):
    """Unify two expressions x and y, returning a substitution dict or None."""
    if subs is None:
        subs = {}

    # Apply existing substitutions
    x = substitute(x, subs)
    y = substitute(y, subs)

    if x == y:
        return subs

    if is_variable(x):
        return unify_var(x, y, subs)
    elif is_variable(y):
        return unify_var(y, x, subs)
    elif isinstance(x, (list, tuple)) and isinstance(y, (list, tuple)) and len(x) == len(y):
        for a, b in zip(x, y):
            subs = unify(a, b, subs)
            if subs is None:
                return None
        return subs
    else:
        return None  # Cannot unify

def unify_var(var, val, subs):
    """Handle unification when var is a variable."""
    if var in subs:
        return unify(subs[var], val, subs)
    elif occurs_check(var, val, subs):
        return None
    else:
        new_subs = subs.copy()
        new_subs[var] = val
        return new_subs

def occurs_check(var, val, subs):
    """Prevent recursive variable bindings."""
    val = substitute(val, subs)
    if var == val:
        return True
    elif isinstance(val, (list, tuple)):
        return any(occurs_check(var, v, subs) for v in val)
    return False

def substitute(expr, subs):
    """Recursively apply substitutions."""
    if is_variable(expr) and expr in subs:
        return substitute(subs[expr], subs)
    elif isinstance(expr, (list, tuple)):
        return type(expr)(substitute(e, subs) for e in expr)
    else:
        return expr

# Example usage
if __name__ == "__main__":
    e1 = ("likes", "x", "icecream")
    e2 = ("likes", "Alice", "y")
    result = unify(e1, e2)
    print("Substitution:", result)


Substitution: {'x': 'Alice', 'icecream': 'y'}


In [6]:
# -------------------- UNIFICATION FUNCTIONS --------------------
from copy import deepcopy

def is_variable(x):
    """A variable starts with '?' (e.g. '?x')"""
    return isinstance(x, str) and x.startswith("?")

def substitute(expr, subs):
    """Recursively apply substitutions"""
    if is_variable(expr) and expr in subs:
        return substitute(subs[expr], subs)
    elif isinstance(expr, (list, tuple)):
        return type(expr)(substitute(e, subs) for e in expr)
    else:
        return expr

def occurs_check(var, val, subs):
    """Prevent recursive definitions (?x = f(?x))"""
    val = substitute(val, subs)
    if var == val:
        return True
    elif isinstance(val, (list, tuple)):
        return any(occurs_check(var, v, subs) for v in val)
    return False

def unify_var(var, val, subs):
    """Unify variable with a value"""
    if var in subs:
        return unify(subs[var], val, subs)
    elif occurs_check(var, val, subs):
        return None
    else:
        new_subs = subs.copy()
        new_subs[var] = val
        return new_subs

def unify(x, y, subs=None):
    """Unify two expressions"""
    if subs is None:
        subs = {}
    x = substitute(x, subs)
    y = substitute(y, subs)
    if x == y:
        return subs
    if is_variable(x):
        return unify_var(x, y, subs)
    elif is_variable(y):
        return unify_var(y, x, subs)
    elif isinstance(x, (list, tuple)) and isinstance(y, (list, tuple)) and len(x) == len(y):
        for a, b in zip(x, y):
            subs = unify(a, b, subs)
            if subs is None:
                return None
        return subs
    return None


# -------------------- FORWARD CHAINING ENGINE --------------------
class KnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.rules = []

    def add_fact(self, fact):
        """Add a single fact (tuple)"""
        self.facts.add(fact)

    def add_rule(self, premises, conclusion):
        """
        premises: list of tuples
        conclusion: single tuple
        """
        self.rules.append((premises, conclusion))

    def infer(self):
        """Forward chaining loop"""
        new_facts = True
        while new_facts:
            new_facts = False
            current_facts = list(self.facts)  # snapshot to avoid mutation errors
            for premises, conclusion in self.rules:
                for subs in self.match_premises(premises, current_facts):
                    new_fact = substitute(conclusion, subs)
                    if new_fact not in self.facts:
                        print(f"Inferred: {new_fact}")
                        self.facts.add(new_fact)
                        new_facts = True

    def match_premises(self, premises, facts_snapshot):
        """Find substitutions that satisfy all premises"""
        def helper(i, current_subs):
            if i == len(premises):
                yield current_subs
            else:
                premise = substitute(premises[i], current_subs)
                for fact in facts_snapshot:
                    subs = unify(premise, fact, deepcopy(current_subs))
                    if subs is not None:
                        yield from helper(i + 1, subs)
        return helper(0, {})


# -------------------- EXAMPLE: ROBERT IS A CRIMINAL --------------------
if __name__ == "__main__":
    kb = KnowledgeBase()

    # --- Facts ---
    kb.add_fact(("American", "Robert"))
    kb.add_fact(("Enemy", "CountryA", "America"))
    kb.add_fact(("Missile", "m1"))
    kb.add_fact(("Owns", "CountryA", "m1"))
    kb.add_fact(("Sells", "Robert", "m1", "CountryA"))

    # --- Rules ---
    # If a country is an enemy of America, it is hostile
    kb.add_rule([("Enemy", "?x", "America")], ("Hostile", "?x"))

    # Missiles are weapons
    kb.add_rule([("Missile", "?x")], ("Weapon", "?x"))

    # It is a crime for an American to sell weapons to hostile nations
    kb.add_rule([
        ("American", "?x"),
        ("Weapon", "?y"),
        ("Sells", "?x", "?y", "?z"),
        ("Hostile", "?z")
    ], ("Criminal", "?x"))

    # --- Run inference ---
    print("Initial facts:")
    for f in kb.facts:
        print(f)
    print("\nDeriving new facts:\n")

    kb.infer()

    print("\nAll facts after inference:")
    for f in kb.facts:
        print(f)

    # --- Print final conclusion ---
    print("\n--- CONCLUSION ---")
    if ("Criminal", "Robert") in kb.facts:
        print("✅ Therefore, Robert is a CRIMINAL.")
    else:
        print("❌ Could not prove that Robert is a criminal.")




Initial facts:
('Enemy', 'CountryA', 'America')
('Sells', 'Robert', 'm1', 'CountryA')
('American', 'Robert')
('Owns', 'CountryA', 'm1')
('Missile', 'm1')

Deriving new facts:

Inferred: ('Hostile', 'CountryA')
Inferred: ('Weapon', 'm1')
Inferred: ('Criminal', 'Robert')

All facts after inference:
('Enemy', 'CountryA', 'America')
('Criminal', 'Robert')
('Hostile', 'CountryA')
('Sells', 'Robert', 'm1', 'CountryA')
('Weapon', 'm1')
('American', 'Robert')
('Owns', 'CountryA', 'm1')
('Missile', 'm1')

--- CONCLUSION ---
✅ Therefore, Robert is a CRIMINAL.
